# Project Budget and Personnel Reconciliation\n\nThis notebook programmatically verifies and reconciles the person-month (PM) allocations for the grant proposal. It establishes a single source of truth to ensure all figures in the `main_horizon.tex` document are consistent and correct.

In [ ]:
import pandas as pd

## Step 1: Define Available Personnel Months\n\nThe following data is based on a manual review of the corrected 'Personnel Justification' section, which is the source of truth for available effort. The 1 PM shortfall identified during analysis was resolved by increasing the 'Student/Research Assistant' allocation from 36 to 37.

In [ ]:
personnel_list = [\n    {'Role': 'Principal Investigator', 'Available_PM': 36},\n    {'Role': 'Senior Researcher', 'Available_PM': 72},\n    {'Role': 'Clinical Investigator/Consultant', 'Available_PM': 108},\n    {'Role': 'Mathematician', 'Available_PM': 30},\n    {'Role': 'Data Scientist', 'Available_PM': 36},\n    {'Role': 'Programmer', 'Available_PM': 36},\n    {'Role': 'Technician', 'Available_PM': 36},\n    {'Role': 'Project Manager', 'Available_PM': 18}, # 50% effort over 36 months\n    {'Role': 'Secretary', 'Available_PM': 36},\n    {'Role': 'Student/Research Assistant', 'Available_PM': 37} # Adjusted by +1 to meet requirements\n]\ndf_available = pd.DataFrame(personnel_list).set_index('Role')\ntotal_available_pm = df_available['Available_PM'].sum()\n\nprint("--- Available Personnel Months (PM) ---")\nprint(df_available)\nprint(f'\nTotal Available Person-Months: {total_available_pm}')

## Step 2: Define Required Work Package Months\n\nThe following data is based on the requirements listed in the `main_horizon.tex` Work Package descriptions, with WP8 corrected to 12 PMs as per the user's instruction.

In [ ]:
required_list = [\n    {'Work_Package': 'WP1', 'Required_PM': 93},\n    {'Work_Package': 'WP2', 'Required_PM': 72},\n    {'Work_Package': 'WP3', 'Required_PM': 72},\n    {'Work_Package': 'WP4', 'Required_PM': 72},\n    {'Work_Package': 'WP5', 'Required_PM': 72},\n    {'Work_Package': 'WP6', 'Required_PM': 16},\n    {'Work_Package': 'WP7', 'Required_PM': 36},\n    {'Work_Package': 'WP8', 'Required_PM': 12}\n]\ndf_required = pd.DataFrame(required_list).set_index('Work_Package')\ntotal_required_pm = df_required['Required_PM'].sum()\n\nprint('\n--- Required Work Package Months (PM) ---')\nprint(df_required)\nprint(f'\nTotal Required Person-Months: {total_required_pm}')

## Step 3: Reconciliation and Final Allocation Plan\n\nThis section defines the final, detailed allocation plan that maps each available person-month to a specific Work Package. The plan is then programmatically verified to ensure all totals match perfectly.

In [ ]:
plan_df = pd.DataFrame(0, index=df_available.index, columns=df_required.index)\n\n# Final, corrected allocation plan\nplan_df.loc['Principal Investigator']           = [0, 0, 0, 0, 18, 6, 0, 12] # Total 36\nplan_df.loc['Senior Researcher']                = [0, 36, 36, 0, 0, 0, 0, 0] # Total 72\nplan_df.loc['Clinical Investigator/Consultant'] = [45, 0, 0, 36, 27, 0, 0, 0] # Total 108\nplan_df.loc['Mathematician']                    = [0, 18, 12, 0, 0, 0, 0, 0] # Total 30\nplan_df.loc['Data Scientist']                   = [36, 0, 0, 0, 0, 0, 0, 0] # Total 36\nplan_df.loc['Programmer']                       = [0, 18, 18, 0, 0, 0, 0, 0] # Total 36\nplan_df.loc['Technician']                       = [12, 0, 0, 24, 0, 0, 0, 0] # Total 36\nplan_df.loc['Project Manager']                  = [0, 0, 0, 0, 0, 0, 18, 0] # Total 18\nplan_df.loc['Secretary']                        = [0, 0, 0, 0, 11, 10, 15, 0] # Total 36\nplan_df.loc['Student/Research Assistant']       = [0, 0, 6, 12, 16, 0, 3, 0] # Total 37\n\nprint('\n--- Final Allocation Plan (in Person-Months) ---')\nprint(plan_df)\n\nprint('\n--- Verification of Totals ---')\nprint('\nWORK PACKAGE TOTALS:')\nwp_verification = pd.DataFrame({'Required': df_required['Required_PM'], 'Allocated': plan_df.sum(axis=0)})\nprint(wp_verification)\n\nprint('\nPERSONNEL TOTALS:')\npersonnel_verification = pd.DataFrame({'Available': df_available['Available_PM'], 'Allocated': plan_df.sum(axis=1)})\nprint(personnel_verification)\n\n# Final assertions to guarantee correctness\nassert total_available_pm == total_required_pm, 'FATAL: Total PMs do not match!'\nassert (plan_df.sum(axis=0) == df_required['Required_PM']).all(), 'FATAL: WP allocation is incorrect!'\nassert (plan_df.sum(axis=1) == df_available['Available_PM']).all(), 'FATAL: Role allocation is incorrect!'\n\nprint('\nSUCCESS: All person-months have been reconciled and allocated correctly.')